# Line Localizers

This notebook explains the different hyperparameters of the PnP+L Localizers and does ablaition studies on some of them. For a rough idea on how to use localizers in general refer to the Quickstart.ipynb

### Hyperparameter Overview:
For environment generation keep in mind, that 3d lines do no further outlier detection then the fitting algorithm, so with noisy scans a high contamination removal can be a good idea.


`extract_and_match_wrapper_config`: How to get the initial PnP based guess, standard is LightGlue

`cam1_line_generator`, `cam2_line_generator`: How to extract the lines from the scan images (cam1) and the query images (cam2). If the second is left as None the one for cam1 will be used for both.

- Each generator takes an `MultiPassLineMergingConfig` consisting of a list of line merging configs that are applied sequentially to the lines.

    - `passes`: multiple line_mergin_configs telling how to cleanup the lines: `max_angle_diff` between 2 lines to be merged. The `max_midpoint_dist` states how colinear 2 lines have to be, increase if colinear chains of lines are not merged. `max_endpoint_dist`, increase if colinear lines are not merged due to high distance along a common line. All lines under `min_line_length` will be removed (all distances and so on normalized to image diagonal length = 1). `use_pca`: wheather to use pca or direction averaging for merging, either produces in very similar results.

- `lsd_diagonal_size`, img size at which Line segment detection should be performed >1000 will cause notable slower inference times.


`line_matching_config`: How to match the 2D lines from the scan image to the ones from the headset image.

- `max_point_line_dist_px`: Maximum distance between a point and a line to be considered near, increase for sparse keypoint / line environments.

- `min_number_supporting_points`: The number of same points that have to be near line1 and line2, to match them. smaller -> more matched lines.

- `better_factor`: A line match has to have least better_factor better more keypoint matches then the second best line pair. Increase -> more FP but also more lines


`line_fitting_3d_config`: How to fit a 2D line segment into 3D, if offline runtime is not crucial just leave at `fitting_algorithm`='ransac'. `min_number_points` can be increaset to accept only longer lines/ ones with more valid supporting points.

`pnpl_optimisation_conf`: Parameters for the pose optimization. `lm_max_steps` defines the max LM-Optimizse steps, `convergence_threshold` the stopping threshold and `line_relevance` the impact of the lines for refinement (higher -> lines are more relevant, 0 -> only point refinement, 1-> only line refinement).

`debug_visualize_pnpl`, `debug_visualize_matching`, `debug_visualize_3d`: Helpful visualizations to understand what is happening. (1. visualizes pose optimization, 2. line matching, 3. the fitted lines in 3d)


### Notebook initialisation

In [ ]:
%load_ext autoreload
%autoreload 2
print(__debug__)

import logging
logging.basicConfig(level=logging.INFO)

import torch, random, os, cv2
import numpy as np

seed = 1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(seed)

import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

import matplotlib.pyplot as plt

### Dataset Loading

In [ ]:
robot_data_folder_location = "../example_datasets/example_small_aruco1"
vrs_file_location = "../example_datasets/small_aruco1_sitting_20fps.vrs"

#robot_data_folder_location = "../example_datasets/small_aruco_2"
#vrs_file_location = "../example_datasets/small_aruco_2_1.vrs"

from shared.complete_robot_scan import CompleteRobotScan
from headset_localization import(
    HeadsetRecording, bind_headset_recording_to_scan, Scanned3dEnvironment, visualize_robot_camera_environment_combo, XYZImageGenerationConfig, ICPAlignmentConfig
) 


robot_data = CompleteRobotScan.from_folder(robot_data_folder_location)
robot_env = Scanned3dEnvironment.from_gathered_robot_data(
        robot_data = robot_data,
        number_of_sampled_datapoints=10,
        sample_datapoints_based_on_aruco_corectness = False,
        only_sample_robot_datapoints_w_marker_estimates = True,
        markers_use_advanced_removal=True,
        est3d_xyz_image_gen_config = XYZImageGenerationConfig(iforest_contamination=0.05, use_depth_images_if_provided=True),
        est3d_xyz_icp_config=ICPAlignmentConfig(do_alginment=True)
)

labeled_headset_data = bind_headset_recording_to_scan(
        headset_data = HeadsetRecording.from_vrs_file(vrs_file_location),
        robot_data = robot_data
)

visualize_loaded_data = False

if visualize_loaded_data:
    visualize_robot_camera_environment_combo(robot_env=robot_env, headset_data=labeled_headset_data)

## Hyperparameters
### Simple Localizer
This shows a simple example localizer on the dataset.

In [ ]:
from headset_localization import *

default_line_predictor = PnPLLocalizer(
        cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
        cam1_bgr_images=robot_env.robot_bgr_images,
        cam1_xyz_images=robot_env.robot_xyz_images,
        extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
            extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
            rotation_augmentations=[Augmentation]
        ),
        cam1_line_generator=LineGenerator(
            visualize_cleanup=False
        ),
        debug_visualize_pnpl=False,
        pnpl_optimisation_conf = PnPLOptimizerConfig(convergence_threshold=1e-8, line_relevance=0.2),
        line_matching_config=LineMatchingConfig(

        ),
        debug_visualize_matching = False,
        debug_visualize_3d = False
)

PredictionOnDataset(predictor = default_line_predictor,headset_data = labeled_headset_data).print_summary()

### Line Generation

This section visualizes the line cleanup with different setups:

In [ ]:
from headset_localization import MultiPassLineMergingConfig, LineMerging2dConfig

INDEX = 0
image = robot_env.robot_bgr_images[INDEX]

# Standard generator
fig, axes = plt.subplots(1,2, figsize = (12, 6))
fig.suptitle("Standard line generator", fontsize=16)
_ = LineGenerator(visualize_cleanup=(axes[0], axes[1])).get_lines(image)


# Fit using weighted direction mean, not pca
fig, axes = plt.subplots(1,2, figsize = (12, 6))
fig.suptitle("Weighted direction mean", fontsize=16)
_ = LineGenerator(
    MultiPassLineMergingConfig(passes=[
        LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850, use_pca=False),
        LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 40/850, use_pca=False)
    ]),
    visualize_cleanup=(axes[0], axes[1])
).get_lines(image)

# Fit using unadjusted connected components
fig, axes = plt.subplots(1,2, figsize = (12, 6))
fig.suptitle("Quick join", fontsize=16)
_ = LineGenerator(
    MultiPassLineMergingConfig(passes=[
        LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850, use_quick_merge = True),
        LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 40/850, use_quick_merge = True)
    ]),
    visualize_cleanup=(axes[0], axes[1])
).get_lines(image)

# Fit at lower resolution
fig, axes = plt.subplots(1,2, figsize = (12, 6))
fig.suptitle("Low res line generator", fontsize=16)
_ = LineGenerator(lsd_diagonal_size=200, visualize_cleanup=(axes[0], axes[1])).get_lines(image)

### 3d Line Fitting Variants
Show effects of different fitting algorithms on performance

In [ ]:
from headset_localization import GradableLocalizer, LineFitting3dConfig, SingleValueErrorType, NPredictors1DatasetGrader

different_3d_fitters = [
    GradableLocalizer(
            creator=PnPLLocalizer.get_creation_function(
                    cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                    line_fitting_3d_config=LineFitting3dConfig(fitting_algorithm=line_fitting_method)
            ),
        category = "PnP+L",
        name=f"{line_fitting_method}"
    )
    for line_fitting_method in ['pca', 'robust-pca', 'ransac']
]

different_3d_fitter_grades = NPredictors1DatasetGrader(
    gradable_pose_predictors=different_3d_fitters,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

fig, ax = plt.subplots(1,1, figsize = (12, 4))
different_3d_fitter_grades.plot_error_vs_error(
    ax=ax, 
    error_type_1=SingleValueErrorType.AVG_TRANSLATIONAL,
    error_type_2=SingleValueErrorType.AVG_ROTATIONAL,
    plot_frontier=False, use_category=False, invert_x=False, invert_y=False, use_in_plot_text = False
)

different_3d_fitter_grades.print_summary()

### Line Relevance 
Effects of the line relevance on performance on the dataset:

In [ ]:
from headset_localization import *

line_relevancies = [
    GradableLocalizer(
        creator=PnPLLocalizer.get_creation_function(
                cam2_intrinsic_mtx=labeled_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    extract_and_match=ExtractAndLightGlue(),
                    ransac_config=pose_estimation_ransaac_config_less_precise,
                    crop_augmentations=[0.2]
                ),
                cam1_line_generator = LineGenerator(visualize_cleanup=False),
                cam2_line_generator=LineGenerator(
                    line_cleanup_config= MultiPassLineMergingConfig(passes=[
                    LineMerging2dConfig(max_angle_diff = 2, max_midpoint_dist = 3/850, max_endpoint_dist = 0.01, min_line_length = 10/850),
                    LineMerging2dConfig(max_angle_diff = 3, max_midpoint_dist = 4/850, max_endpoint_dist = 0.02, min_line_length = 15/850)
                    ]),
                    visualize_cleanup=False
                ),
                pnpl_optimisation_conf=PnPLOptimizerConfig(
                    line_relevance=line_relevance
                ),
                debug_visualize_pnpl = False
            ),
        category = "Points & Lines:",
        name=f"LR: {line_relevance:.2f}",
        value=line_relevance,
    )
    for line_relevance in np.arange(0, 0.8, 0.05)
]

Now those can be used to create a grader object for multiple `PosePredictor` variants.

In [ ]:
line_relevance_grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=line_relevancies,
    headset_data = labeled_headset_data,
    robot_env = robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

In [ ]:
fig, axes1 = plt.subplots(1,2, figsize = (12, 1.5))
line_relevance_grader.plot_value_vs_error(
    ax = axes1[0],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.AVG_TRANSLATIONAL,
    use_category=True
)

line_relevance_grader.plot_value_vs_error(
    ax = axes1[1],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.AVG_ROTATIONAL,
    use_category=True
)

fig, axes1 = plt.subplots(1,2, figsize = (12, 1.5))
line_relevance_grader.plot_value_vs_error(
    ax = axes1[0],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line relevancy",
    error_type = SingleValueErrorType.MED_TRANSLATIONAL,
    use_category=True
)

line_relevance_grader.plot_value_vs_error(
    ax = axes1[1],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line relevancy",
    error_type = SingleValueErrorType.MED_ROTATIONAL,
    use_category=True
)

fig, ax = plt.subplots(1,1, figsize = (8, 8))
line_relevance_grader.plot_prediction_times(ax)
plt.show()

Effects of the line relevance on the first 10% of the fr2/desk dataset

In [ ]:
from headset_localization import *

tum_rgbd_dataset_location = "../tum_datasets/rgbd_dataset_freiburg2_desk"

tum_robot_env, tum_headset_data = scanned_3d_environment_and_headset_recording_from_tum(
        folder=tum_rgbd_dataset_location,
        rgb_camera_name="freiburg2",
        time_tolerance= 0.03,
        n_robot_images= 20,
        xyz_image_generation_config=XYZImageGenerationConfig(iforest_contamination = 0.4, use_depth_images_if_provided=False),
        xyz_image_alginment_config=ICPAlignmentConfig(do_alginment=True),
        intervall=(0.0, 0.1)
)

In [ ]:
line_relevancies_tum = [
    GradableLocalizer(
        creator=PnPLLocalizer.get_creation_function(
                cam2_intrinsic_mtx=tum_headset_data.intrinsic_cam_mtx,
                extract_and_match_wrapper_config=ExtractAndMatchWrapperConfig(
                    rotation_augmentations = [Augmentation],
                    extract_and_match=ExtractAndMatchLoMa(loma_variant="LoMaB128"),
                    ransac_config=pose_estimation_ransaac_config_less_precise,
                ),
                pnpl_optimisation_conf=PnPLOptimizerConfig(
                    line_relevance=line_relevance
                ),
                line_matching_config = LineMatchingConfig(
                    min_number_supporting_points = 2,
                    better_factor=1.5
                ),
                debug_visualize_pnpl = False,
                debug_visualize_3d = False,
                debug_visualize_matching = False
            ),
        category = "Points & Lines:",
        name=f"LR: {line_relevance:.2f}",
        value=line_relevance,
    )
    for line_relevance in np.arange(0, 0.8, 0.05)
]

line_relevance_grader = NPredictors1DatasetGrader(
    gradable_pose_predictors=line_relevancies_tum,
    headset_data = tum_headset_data,
    robot_env = tum_robot_env,
    use_tqdm_for_predictors=True,
    use_tqdm_for_frames=False
)

In [ ]:
line_relevance_grader.print_summary()

fig, axes1 = plt.subplots(1,2, figsize = (12, 1.5))
line_relevance_grader.plot_value_vs_error(
    ax = axes1[0],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.AVG_TRANSLATIONAL,
    use_category=True
)

line_relevance_grader.plot_value_vs_error(
    ax = axes1[1],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.AVG_ROTATIONAL,
    use_category=True
)

fig, axes1 = plt.subplots(1,2, figsize = (12, 1.5))
line_relevance_grader.plot_value_vs_error(
    ax = axes1[0],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.MED_TRANSLATIONAL,
    use_category=True
)

line_relevance_grader.plot_value_vs_error(
    ax = axes1[1],
    x_axis_label= "Line relevancy",
    x_axis_title_name = "Line importance",
    error_type = SingleValueErrorType.MED_ROTATIONAL,
    use_category=True
)
plt.show()